# 01 Data Loading

This notebook contains **Steps 1A–1E** of the project workflow:

- **1A**: environment setup / library installation
- **1B**: optional Google Drive mount for Colab
- **1C**: project path setup and CSV verification
- **1D**: load CSV files into DuckDB
- **1E**: verify loaded tables

**GitHub note:** this notebook preserves the original workflow, but the Google Drive mount is only needed in Colab. For local or GitHub-based use, replace the project path with your local data directory.


# Install Required Libraries





In [1]:
# Run this only if the required packages are not already installed.
# In local environments, prefer installing from requirements.txt.
!pip install duckdb pandas pyarrow


* We have installed required libraries to create a database and store all the
dataset files for faster access.

# Step 1A: Set Data Path and Verify Files

In [2]:
import os

DATA_PATH = os.path.join(os.path.dirname(os.path.dirname(os.path.abspath("__file__"))), "dataset") + "/"

print(os.listdir(DATA_PATH))

['admissions.csv', 'heart_diagnoses.csv', 'heart_diagnoses_all.csv', 'heart_diagnoses_all_true.csv', 'heart_labevents_examination_group.csv', 'heart_labevents_first_lab.csv', 'heart_microbiologyevents.csv', 'heart_microbiologyevents_first_micro.csv', 'heart_procedures.csv', 'hf_project.duckdb', 'HPI.json', 'patients.csv', 'RAG_data_summary.json']


* Defined the project data path and lists all available CSV files and database to confirm the dataset is accessible.

# Step 1B: Loaded All CSV Files into DuckDB Database

In [3]:
import duckdb
import os

DATA_PATH = os.path.join(os.path.dirname(os.path.dirname(os.path.abspath("__file__"))), "dataset") + "/"

from utils import get_db_connection
con = get_db_connection()

csv_files = [f for f in os.listdir(DATA_PATH) if f.endswith(".csv")]

for file in csv_files:
    table_name = file.replace(".csv", "")
    print(f"Loading {file} as table: {table_name}")

    con.execute(f"""
        CREATE OR REPLACE TABLE {table_name} AS
        SELECT * FROM read_csv_auto('{DATA_PATH}{file}', IGNORE_ERRORS=true);
    """)

print('Database created at the configured path')

Connected to DuckDB at: d:\Capstone project\Capstone_Healthcare_Decision_Intelligence_Agent\dataset\hf_project.duckdb
Loading admissions.csv as table: admissions
Loading heart_diagnoses.csv as table: heart_diagnoses
Loading heart_diagnoses_all.csv as table: heart_diagnoses_all
Loading heart_diagnoses_all_true.csv as table: heart_diagnoses_all_true
Loading heart_labevents_examination_group.csv as table: heart_labevents_examination_group
Loading heart_labevents_first_lab.csv as table: heart_labevents_first_lab
Loading heart_microbiologyevents.csv as table: heart_microbiologyevents
Loading heart_microbiologyevents_first_micro.csv as table: heart_microbiologyevents_first_micro
Loading heart_procedures.csv as table: heart_procedures
Loading patients.csv as table: patients
Database created at the configured path


* Connected to a DuckDB database on Google Drive and loads all 11
CSV files as tables using `read_csv_auto`. This creates a local analytical database for efficient SQL-based querying throughout the project.

# Step 1C: Verify Loaded Tables

In [4]:
con.execute("SHOW TABLES").fetchdf()


,name
0,admissions
1,heart_diagnoses
2,heart_diagnoses_all
3,heart_diagnoses_all_true
4,heart_labevents_examination_group
5,heart_labevents_first_lab
6,heart_microbiologyevents
7,heart_microbiologyevents_first_micro
8,heart_procedures
9,patients


Listed all tables currently stored in the DuckDB database to confirm successful data loading. The database contains 15 tables including raw data tables and derived feature tables from previous pipeline steps.

In [5]:
con.close()